In [31]:

import cv2
import os
import re
import numpy as np
import math
import random
from IPython.display import Video, display
from google.colab.patches import cv2_imshow
from base64 import b64encode
from IPython.display import HTML, display
import statistics
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Carregando os videos

In [47]:
import os
from pathlib import Path


def carregar_video(caminho_da_pasta):
    pasta = Path(caminho_da_pasta)

    if not pasta.exists() or not pasta.is_dir():
        print(f"Erro: O caminho '{caminho_da_pasta}' não é válido.")
        return []

    arquivos_png = sorted(
        [f for f in pasta.iterdir() if f.is_file() and f.suffix.lower() == ".png"]
    )

    print(f"Sucesso! {len(arquivos_png)} arquivos .png encontrados e ordenados.")
    return arquivos_png


caminhos_treino = [
    "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/1_Times_Square/View_1/Train",
    "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/1_Times_Square/View_2/Train",
    "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/1_Times_Square/View_3/Train",
    "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/2_Las_Vegas/Train",
    "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/3_Love Parade/Train_1",
    "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/3_Love Parade/Train_2"
]
caminhos_teste = [
    "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/1_Times_Square/View_1/Test",
    "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/1_Times_Square/View_2/Test",
    "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/1_Times_Square/View_3/Test",
    "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/2_Las_Vegas/Test_1",
    "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/2_Las_Vegas/Test_2",
    "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/2_Las_Vegas/Test_3",
    "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/2_Las_Vegas/Test_4",
    "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/3_Love Parade/Test",
    "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/4_Italy/View_1/Test",
    "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/4_Italy/View_2/Test"
]
#caminhos_treino = ["/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/1_Times_Square/View_1/Train"]

X_train_frames_paths = [carregar_video(caminhos_treino[i]) for i in range(len(caminhos_treino))]
X_test_frames_paths = [carregar_video(caminhos_teste[i]) for i in range(len(caminhos_teste))]

#caminhos_teste = "/kaggle/input/datasets/angelchi56/abnormal-highdensity-crowds/Abnormal High-density Crowds/1_Times_Square/View_2/Train"

def ler_frame(caminho_do_frame):
    frame = cv2.imread(str(caminho_do_frame))

    if frame is None:
        print(f"Erro ao carregar o frame: {caminho_do_frame}")
        return None

    return frame


Sucesso! 379 arquivos .png encontrados e ordenados.
Sucesso! 150 arquivos .png encontrados e ordenados.
Sucesso! 184 arquivos .png encontrados e ordenados.
Sucesso! 4347 arquivos .png encontrados e ordenados.
Sucesso! 131 arquivos .png encontrados e ordenados.
Sucesso! 119 arquivos .png encontrados e ordenados.
Sucesso! 1026 arquivos .png encontrados e ordenados.
Sucesso! 1173 arquivos .png encontrados e ordenados.
Sucesso! 1151 arquivos .png encontrados e ordenados.
Sucesso! 1062 arquivos .png encontrados e ordenados.
Sucesso! 920 arquivos .png encontrados e ordenados.
Sucesso! 1854 arquivos .png encontrados e ordenados.
Sucesso! 3407 arquivos .png encontrados e ordenados.
Sucesso! 361 arquivos .png encontrados e ordenados.
Sucesso! 702 arquivos .png encontrados e ordenados.
Sucesso! 702 arquivos .png encontrados e ordenados.


# Visualização do Video

In [48]:
import cv2
from pathlib import Path


def criar_video_dos_pngs(imagens, caminho_video_saida, fps=30):
    if not imagens:
        print("Erro: A lista de imagens está vazia.")
        return

    primeira_img = cv2.imread(str(imagens[0]))
    altura, largura, camadas = primeira_img.shape
    tamanho = (largura, altura)

    quatrocc = cv2.VideoWriter_fourcc(*"mp4v")
    video = cv2.VideoWriter(caminho_video_saida, quatrocc, fps, tamanho)

    for caminho_img in imagens:
        img = cv2.imread(str(caminho_img))
        video.write(img)

    video.release()
    print(f"Vídeo salvo com sucesso em: {caminho_video_saida}")


caminho_train = "/kaggle/working/train.mp4"
caminho_test = "/kaggle/working/test.mp4"
criar_video_dos_pngs(X_train_frames_paths[2], caminho_train, fps=30)
criar_video_dos_pngs(X_test_frames_paths[2], caminho_test, fps=30)

Vídeo salvo com sucesso em: /kaggle/working/train.mp4


KeyboardInterrupt: 

# Treinamento do Modelo

In [55]:
def convert_video_to_grayscale(video):
  gray_frames = []
  for i in range(len(video)):
    frame = video.get_frame(i)
    gray = cv2.cvtColor(
		  frame,
		  cv2.COLOR_BGR2GRAY
	  )
    gray_frames.append(gray)
  return gray_frames

def optical_flow(gray1, gray2):
	return cv2.calcOpticalFlowFarneback(
		gray1,
		gray2,
		None,
		pyr_scale=0.5,
		levels=3,
		winsize=15,
		iterations=3,
		poly_n=5,
		poly_sigma=1.2,
		flags=0
	)

def mostrar_imagem_kaggle(imagem, titulo="Imagem"):
    """
    Renderiza uma imagem do OpenCV (numpy array) diretamente no Kaggle Notebook.
    """
    # 1. Converte de BGR (padrão do OpenCV) para RGB (padrão do Matplotlib)
    imagem_rgb = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)
    
    # 2. Configura o tamanho da exibição no notebook
    plt.figure(figsize=(8, 6))
    
    # 3. Renderiza a imagem e remove os eixos (opcional, mas deixa visualmente limpo)
    plt.imshow(imagem_rgb)
    plt.title(titulo)
    plt.axis('off') 
    
    # 4. Mostra o plot
    plt.show()

import cv2
import numpy as np
import matplotlib.pyplot as plt

def draw_optical_flow(frame1, frame2, step=16, scale_factor=3):
    """
    Calcula o Optical Flow e desenha vetores amplificados para visibilidade.
    
    scale_factor: Aumenta o tamanho visual das setinhas. 
                  Aumente se ainda não estiver vendo as setas.
    """
    flow = cv2.calcOpticalFlowFarneback(
        frame1,
        frame2,
        None,
        pyr_scale=0.5,
        levels=3,
        winsize=15,
        iterations=3,
        poly_n=5,
        poly_sigma=1.2,
        flags=0
    )

    flow_vis = cv2.cvtColor(
        frame1,
        cv2.COLOR_GRAY2BGR
    )

    h, w = frame1.shape

    for y in range(0, h, step):
        for x in range(0, w, step):
            fx, fy = flow[y, x]
            
            # --- Correção aqui: Amplificar o deslocamento visual ---
            x2 = int(x + fx * scale_factor)
            y2 = int(y + fy * scale_factor)

            # Desenha a setinha verde
            cv2.arrowedLine(
                flow_vis,
                (x, y),
                (x2, y2),
                (0, 255, 0),
                1,
                tipLength=0.3
            )

            # Desenha o ponto vermelho
            cv2.circle(
                flow_vis,
                (x, y),
                1,
                (0, 0, 255),
                -1
            )

    plt.figure(figsize=(12, 10)) # Aumentei um pouco o tamanho da figura
    plt.imshow(cv2.cvtColor(flow_vis, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

def converte_frame(frame: np.ndarray, largura=384, altura=288) -> np.ndarray:
    # converte para escala de cinza (1 canal) se o frame for colorido
    if len(frame.shape) == 3:
        frame_processado = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        frame_processado = frame.copy()

    # redimensiona
    frame_processado = cv2.resize(frame_processado, (largura, altura), interpolation=cv2.INTER_AREA)

    return frame_processado

def aplicar_invariancia_direcao(janela_temporal):
    soma_direcoes = np.sum(janela_temporal, axis=0)

    i_direcao_dominante = np.argmax(soma_direcoes)

    shift_val = -i_direcao_dominante

    return np.roll(janela_temporal, shift=shift_val, axis=1)

import numpy as np

def extrair_amostras_treinamento(dataset, window_size: int = 50):
    """
    Para cada video, gera os H.O.D para cada par de frames do video e normaliza
    os valores para [0,1].
    Esses histogramas são então aplicados a uma função que identifica a direção
    dominante de cada janela e a coloca como posição 0.
    A saída da função é uma matriz 50x8 pelo total de janelas temporais gerados,
    que mostra a evolução dos H.O.D.s ao longo de uma dada sequência.
    """
    X = []

    for i in range(len(dataset)):
        video = dataset[i]
        num_frames = len(video)

        histogramas_do_video = []

        for f in range(num_frames - 1):
            frame1 = ler_frame(video[f])
            frame2 = ler_frame(video[f+1])
            
            convertido1 = converte_frame(frame1)
            convertido2 = converte_frame(frame2)
            
            # Calcula o optical flow entre os frames
            flow = optical_flow(convertido1, convertido2)
            # Matrizes de deslocamento horizontal e vertical entre os frames
            fx = flow[..., 0]
            fy = flow[..., 1]

            magnitude = np.sqrt(fx**2 + fy**2)
            direcao = np.arctan2(fy, fx)

            #pixels_movendo = magnitude > 0.5
            #angulos_validos = direcao[pixels_movendo]

            # H.O.D
            hist_frame, _ = np.histogram(direcao, bins=8, range=(-np.pi, np.pi))
            if (f == 0):
                print(hist_frame)
            # Normalização do histograma pra [0,1]
            total_votos = np.sum(hist_frame)
            if total_votos > 0:
                hist_frame = hist_frame / total_votos
            if (f == 0):
                print(hist_frame)
            histogramas_do_video.append(hist_frame)

        # Shape: (N_frames-1, 8)
        histogramas_do_video = np.array(histogramas_do_video)
        num_histogramas = len(histogramas_do_video)
        print(f"N de Histogramas do video: {i}: {num_histogramas}")
        
        for j in range(num_histogramas - window_size + 1):
            # Janela deslizante
            # Pega histogramas da janela atual 
            # Shape: (50,8)
            janela_bruta = histogramas_do_video[j : j + window_size] 
            # Aplica invariância a direção
            janela_invariante = aplicar_invariancia_direcao(janela_bruta) 
            X.append(janela_invariante)

    return np.array(X)

def motion_directional_pca(X_janelas_treino):
	len_hist = X_janelas_treino.shape[2]
	matrizes_reduzidas = []
	pcas_treinados = []
	for c in range(len_hist):
		# Janela no formato (N, T)
		janela = X_janelas_treino[:,:,c]
		# PCA
		pca = PCA(n_components=0.95)
		matriz_reduzida = pca.fit_transform(janela)
		# Salva o vetor de features reduzido
		matrizes_reduzidas.append(matriz_reduzida)
		# Salva o PCA treinado
		pcas_treinados.append(pca)

	X_treino_final = np.hstack(matrizes_reduzidas)
	return X_treino_final, pcas_treinados

def train_model(dataset, T=50, D=8):
	print("Gerando as janelas invariantes à direção.")
	amostras = extrair_amostras_treinamento(dataset, T)
	print("Aplicando Motion directional PCA para reduzir a dimensionalidade das janelas")
	X, pcas_treinados = motion_directional_pca(amostras)
	print("Treinando o modelo")
	detector_anomalia = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05)
	detector_anomalia.fit(X)
	return detector_anomalia, pcas_treinados


In [56]:
detector, pcas = train_model(X_train_frames_paths)

Gerando as janelas invariantes à direção.
[14244 12117 13128 18123  9390 12237 16288 15065]
[0.12879774 0.10956489 0.1187066  0.16387261 0.08490668 0.11064996
 0.14728009 0.13622143]
N de Histogramas do video: 0: 378
[12572  9012 11407 21332 23151 10326 10908 11884]
[0.11367911 0.08148872 0.10314489 0.19288918 0.20933702 0.09337023
 0.09863281 0.10745804]
N de Histogramas do video: 1: 149
[13612 14382 14444 13142 14545 14144 14003 12320]
[0.12308304 0.13004557 0.13060619 0.11883319 0.13151946 0.12789352
 0.12661856 0.11140046]
N de Histogramas do video: 2: 183
[10663 10084 13321 25734 22399 11533  7684  9174]
[0.09641746 0.091182   0.12045175 0.23269314 0.20253725 0.10428422
 0.06948061 0.08295356]
N de Histogramas do video: 3: 4346
[14939  6187  8273 12287 15877 14641 16076 22312]
[0.1350821  0.05594437 0.0748065  0.11110207 0.14356373 0.13238751
 0.14536314 0.20175058]
N de Histogramas do video: 4: 130
[42727  5472  1991   861  1044  1803  4881 51813]
[0.38634802 0.04947917 0.0180031

# Testes do Modelo

In [57]:
def motion_directional_pca_teste(X_janelas, pcas_treinados):
    qtd_direcoes = X_janelas.shape[2]
    matrizes_reduzidas = []

    for c in range(qtd_direcoes):
        janela = X_janelas[:, :, c]
        pca = pcas_treinados[c]
        matriz_reduzida = pca.transform(janela)
        matrizes_reduzidas.append(matriz_reduzida)

    X_final = np.hstack(matrizes_reduzidas)
    return X_final

def extrair_amostras_teste(video, window_size: int = 50):
    X = []
    num_frames = len(video)

    histogramas_do_video = []

    for f in range(num_frames - 1):
        frame1 = ler_frame(video[f])
        frame2 = ler_frame(video[f+1])

        convertido1 = converte_frame(frame1)
        convertido2 = converte_frame(frame2)

        flow = optical_flow(convertido1, convertido2)

        fx = flow[..., 0]
        fy = flow[..., 1]

        magnitude = np.sqrt(fx**2 + fy**2)
        direcao = np.arctan2(fy, fx)

        #pixels_movendo = magnitude > 0.5
        #angulos_validos = direcao[pixels_movendo]

        hist_frame, _ = np.histogram(direcao, bins=8, range=(-np.pi, np.pi))

        total_votos = np.sum(hist_frame)
        if total_votos > 0:
            hist_frame = hist_frame / total_votos

        histogramas_do_video.append(hist_frame)

    histogramas_do_video = np.array(histogramas_do_video)
    num_histogramas = len(histogramas_do_video)

    for j in range(num_histogramas - window_size + 1):
        janela_bruta = histogramas_do_video[j : j + window_size]
        janela_invariante = aplicar_invariancia_direcao(janela_bruta)
        X.append(janela_invariante)

    return np.array(X)

def inferencia(video, detector, pcas_treinados):
  amostras = extrair_amostras_teste(video)
  X = motion_directional_pca_teste(amostras, pcas_treinados)
  return detector.predict(X)

In [60]:
def acuracia(dataset_teste, detector, pcas, window_size=50):
    y_true = []
    y_pred = []
    for i in range(len(dataset_teste)):
        y_true_video = []
        y_pred_video = []
        
        video = dataset_teste[i]
        predict_windows = inferencia(video, detector, pcas)
        print(f"Avaliando video {i}")
        num_frames = len(video)
        for j in range(len(predict_windows)):
            label_window = 1 # Dataset de teste só contem frames anomalos
            y_true_video.append(label_window)
            y_pred_video.append(predict_windows[j])
            print(f"Janela {j} video {i}. Label: {label_window}, predict:{predict_windows[j]}")

        print(f"Acurácia video {i}: {accuracy_score(y_true, y_pred)}")
        y_true.extend(y_true_video)
        y_pred.extend(y_pred_video)

    accuracy = accuracy_score(y_true, y_pred)
    return accuracy

print(acuracia(X_test_frames_paths, detector, pcas))

Avaliando video 0
Janela 0 video 0. Label: 1, predict:1
Janela 1 video 0. Label: 1, predict:1
Janela 2 video 0. Label: 1, predict:1
Janela 3 video 0. Label: 1, predict:1
Janela 4 video 0. Label: 1, predict:1
Janela 5 video 0. Label: 1, predict:1
Janela 6 video 0. Label: 1, predict:1
Janela 7 video 0. Label: 1, predict:1
Janela 8 video 0. Label: 1, predict:1
Janela 9 video 0. Label: 1, predict:1
Janela 10 video 0. Label: 1, predict:1
Janela 11 video 0. Label: 1, predict:1
Janela 12 video 0. Label: 1, predict:1
Janela 13 video 0. Label: 1, predict:1
Janela 14 video 0. Label: 1, predict:1
Janela 15 video 0. Label: 1, predict:1
Janela 16 video 0. Label: 1, predict:1
Janela 17 video 0. Label: 1, predict:1
Janela 18 video 0. Label: 1, predict:1
Janela 19 video 0. Label: 1, predict:1
Janela 20 video 0. Label: 1, predict:1
Janela 21 video 0. Label: 1, predict:1
Janela 22 video 0. Label: 1, predict:1
Janela 23 video 0. Label: 1, predict:1
Janela 24 video 0. Label: 1, predict:1
Janela 25 video 0

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:557: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Avaliando video 1
Janela 0 video 1. Label: 1, predict:1
Janela 1 video 1. Label: 1, predict:1
Janela 2 video 1. Label: 1, predict:1
Janela 3 video 1. Label: 1, predict:1
Janela 4 video 1. Label: 1, predict:1
Janela 5 video 1. Label: 1, predict:1
Janela 6 video 1. Label: 1, predict:1
Janela 7 video 1. Label: 1, predict:1
Janela 8 video 1. Label: 1, predict:1
Janela 9 video 1. Label: 1, predict:1
Janela 10 video 1. Label: 1, predict:-1
Janela 11 video 1. Label: 1, predict:-1
Janela 12 video 1. Label: 1, predict:1
Janela 13 video 1. Label: 1, predict:1
Janela 14 video 1. Label: 1, predict:-1
Janela 15 video 1. Label: 1, predict:1
Janela 16 video 1. Label: 1, predict:-1
Janela 17 video 1. Label: 1, predict:1
Janela 18 video 1. Label: 1, predict:1
Janela 19 video 1. Label: 1, predict:-1
Janela 20 video 1. Label: 1, predict:-1
Janela 21 video 1. Label: 1, predict:1
Janela 22 video 1. Label: 1, predict:-1
Janela 23 video 1. Label: 1, predict:1
Janela 24 video 1. Label: 1, predict:1
Janela 25 

# Visualização do Modelo

In [21]:

import os
import re
import cv2
import numpy as np

def inferencia_completa_e_gerar_video(video, detector, pcas_treinados, video_number, window_size=50, output_fps=25):
    """
    Executa o pipeline completo de teste para um único vídeo, renderizando
    a predição e o score numérico da decision_function nos frames.
    """
    print(f"\n🎬 Iniciando pipeline visual (com Scores) para o Vídeo {video_number:02d}.")

    # =================================================================
    # PASSO 1: Extração, Classificação e Scores (Memória RAM)
    # =================================================================
    amostras = extrair_amostras_teste(video, window_size)

    if len(amostras) == 0:
        print("Erro: O vídeo é muito curto para gerar janelas temporais.")
        return None

    X_teste_final = motion_directional_pca_teste(amostras, pcas_treinados)

    # Pegamos o veredito (-1 ou 1) E a distância contínua até a fronteira
    predicoes = detector.predict(X_teste_final)
    scores = detector.decision_function(X_teste_final)

    # =================================================================
    # PASSO 2: Renderização dos Status e Scores nos Frames
    # =================================================================
    num_frames_totais = len(video)
    frames_processados = []

    # Preenche as primeiras frames com status neutro (sem score ainda)
    for f in range(window_size):
        frame_original = ler_frame(video[f]).copy()
        cv2.rectangle(frame_original, (10, 10), (320, 55), (0, 0, 0), -1)
        cv2.putText(frame_original, "ANALISANDO...", (20, 42),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 165, 0), 2, cv2.LINE_AA)
        frames_processados.append(frame_original)

    # Vincula o resultado e o score de cada janela ao frame de fechamento
    for j in range(len(predicoes)):
        frame_idx = j + window_size
        if frame_idx >= num_frames_totais:
            break

        frame_atual = ler_frame(video[frame_idx]).copy()
        resultado_janela = predicoes[j]
        score_janela = scores[j] # Distância matemática da janela atual

        # Define a cor e o texto principal com base na predição
        if resultado_janela == 1:
            status_txt = "NORMAL"
            cor = (0, 255, 0)  # Verde BGR
        else:
            status_txt = "ANOMALIA"
            cor = (0, 0, 255)  # Vermelho BGR

        # Monta a string final contendo o Status e o Score formatado
        texto_completo = f"{status_txt} | Score: {score_janela:.4f}"

        # Retângulo preto de fundo ligeiramente maior para acomodar o score largo
        cv2.rectangle(frame_atual, (10, 10), (450, 55), (0, 0, 0), -1)

        # Renderiza a string unificada
        cv2.putText(
            frame_atual,
            texto_completo,
            (20, 42),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,  # Fonte ligeiramente menor para caber tudo na caixa
            cor,
            2,
            cv2.LINE_AA
        )

        frames_processados.append(frame_atual)

    # =================================================================
    # PASSO 3: Geração do MP4 Otimizado para Navegador
    # =================================================================
    video_str = f"{video_number:02d}"
    height, width, _ = frames_processados[0].shape
    size = (width, height)

    temp_output = f"video_{video_str}_anotado_temp.mp4"
    final_output = f"video_{video_str}_anotado.mp4"

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(temp_output, fourcc, output_fps, size)

    print(f"Compilando {len(frames_processados)} frames anotados...")
    for img in frames_processados:
        video_writer.write(img)
    video_writer.release()

    print("Otimizando codec para H.264 (Compatibilidade com Navegadores) via FFmpeg...")
    os.system(f"ffmpeg -y -i {temp_output} -vcodec libx264 -pix_fmt yuv420p {final_output} -loglevel quiet")

    if os.path.exists(temp_output):
        os.remove(temp_output)

    print(f"🎉 Concluído! Vídeo com scores gerado em: {final_output}")
    return final_output

video = X_test_frames_paths[2]
inferencia_completa_e_gerar_video(video, detector, pcas, 3)


🎬 Iniciando pipeline visual (com Scores) para o Vídeo 03.
Compilando 1151 frames anotados...
Otimizando codec para H.264 (Compatibilidade com Navegadores) via FFmpeg...
🎉 Concluído! Vídeo com scores gerado em: video_03_anotado.mp4


'video_03_anotado.mp4'